# Sweden CBA / PBA hotspot analysis — EXIOBASE 3 v3.8.2

**Project:** RISE pre-study for Region Stockholm. This notebook is the stable national baseline. A later step will layer Stockholm / Rest-of-Sweden disaggregation on top.

**What this notebook does.** A national-level hotspot analysis of Sweden across three dimensions:

1. **Economic importance** — value added (M EUR)
2. **GHG emissions** — fossil CO2-eq (kt), with a separate biogenic CO2-eq column
3. **Material footprint** — kt, aggregated into the four Anthesis primary categories (biomass, fossil, metals, minerals)

For each dimension we compute production-based (PBA) and consumption-based (CBA) accounts, identify top sectors, attribute Sweden's CBA to its source countries and sectors, and trace Sweden's PBA to where it is ultimately consumed.

**Database:** EXIOBASE 3 v3.8.2 (`IOT_2022_pxp.zip`) — free for commercial use.

**Outputs** are written to `./outputs/` (CSV tables and chart PNGs / HTML).


## 1. Setup

In [1]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

import pymrio

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200,
                     "axes.titlesize": 11, "axes.labelsize": 9,
                     "xtick.labelsize": 8, "ytick.labelsize": 8})


In [2]:
# ---------- Path configuration -------------------------------------------
EXIOBASE_PATH = r"C:\EXIOBASE3\IOT_2022_pxp.zip"

# ---------- Focus region and analysis knobs -------------------------------
REGION = "SE"          # ISO code for Sweden in EXIOBASE
TOP_N  = 15            # how many entries to show in top-N tables / charts

OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# ---------- Unit conventions (reference) ---------------------------------
# Raw EXIOBASE v3.8.2 units:
#   economic core  : M.EUR  (current prices)
#   air_emissions  : kg     (physical except HFC/PFC which are kg CO2-eq; SF6 physical)
#   material       : kt
#   factor_inputs  : M.EUR
#
# Reporting units used in this notebook:
#   economic       : M.EUR (value added)
#   GHG            : kt CO2-eq     (kg -> kt: divide by 1e6)
#   material       : kt

print(f"EXIOBASE path  : {EXIOBASE_PATH}")
print(f"Focus region   : {REGION}")
print(f"Top-N          : {TOP_N}")
print(f"Output folder  : {OUTPUT_DIR.resolve()}")


EXIOBASE path  : C:\EXIOBASE3\IOT_2022_pxp.zip
Focus region   : SE
Top-N          : 15
Output folder  : C:\Users\rafaella\OneDrive - RISE\Region Stockholm - Documents\Cirkulär ekonomi\5 Arbetsmaterial och underlag\Dimension_123\outputs


## 2. Load EXIOBASE

We parse the archive with `pymrio.parse_exiobase3`. In v3.8.2 the pre-computed extension multipliers (`S`, `M`) and footprint accounts (`D_cba`, `D_pba`) are present in the archive, which saves one expensive computation step for us.

In [3]:
def load_exio(path):
    print(f"Parsing EXIOBASE from: {path}")
    t0 = time.time()
    exio = pymrio.parse_exiobase3(path=path)
    print(f"  parsed in {time.time()-t0:.1f}s")
    print(f"  A shape : {None if exio.A is None else exio.A.shape}")
    print(f"  Y shape : {exio.Y.shape}")
    print(f"  x shape : {None if exio.x is None else exio.x.shape}")
    regions = exio.get_regions().tolist()
    sectors = exio.get_sectors().tolist()
    print(f"  regions : {len(regions)} ({regions[:5]}...)")
    print(f"  sectors : {len(sectors)}")
    for ext_name in ['air_emissions', 'material', 'factor_inputs', 'impacts']:
        ext = getattr(exio, ext_name, None)
        if ext is not None and ext.F is not None:
            S_ok = ext.S is not None
            M_ok = ext.M is not None
            print(f"  ext '{ext_name}': F={ext.F.shape}, S={'Y' if S_ok else 'N'}, M={'Y' if M_ok else 'N'}")
        else:
            print(f"  ext '{ext_name}': NOT FOUND")
    return exio

exio = load_exio(EXIOBASE_PATH)


Parsing EXIOBASE from: C:\EXIOBASE3\IOT_2022_pxp.zip
  parsed in 92.8s
  A shape : (9800, 9800)
  Y shape : (9800, 343)
  x shape : (9800, 1)
  regions : 49 (['AT', 'BE', 'BG', 'CY', 'CZ']...)
  sectors : 200
  ext 'air_emissions': NOT FOUND
  ext 'material': NOT FOUND
  ext 'factor_inputs': NOT FOUND
  ext 'impacts': F=(126, 9800), S=Y, M=Y


## 3. Compute the Leontief inverse L

`calc_system()` computes `A = Z · diag(x)⁻¹` (if not already present) and the Leontief inverse `L = (I − A)⁻¹`. This is the single memory-heavy step (~3 GB peak on the 9800×9800 pxp system). We call it only once per kernel session.

In [4]:
if getattr(exio, "L", None) is None:
    print("Computing Leontief inverse L (can take a few minutes)...")
    t0 = time.time()
    exio.calc_system()
    print(f"  L shape: {exio.L.shape}   ({time.time()-t0:.1f}s)")
else:
    print(f"L already computed: {exio.L.shape}. Skipping.")


Computing Leontief inverse L (can take a few minutes)...
  L shape: (9800, 9800)   (18.2s)


## 4. Indicator helpers

### 4.1 Region slicing

In [5]:
def cols_of(df, region):
    """Select columns of a (region, sector)-indexed DataFrame for one region."""
    return df.loc[:, pd.IndexSlice[region, :]]

def rows_of(df, region):
    """Select rows of a (region, sector)-indexed DataFrame for one region."""
    return df.loc[pd.IndexSlice[region, :], :]

# Sanity check:
_se_sectors = cols_of(exio.Y, REGION).columns.get_level_values(1).tolist()
print(f"Sweden has {len(_se_sectors)} product sectors. First 5:")
for s in _se_sectors[:5]:
    print(f"  {s}")


Sweden has 7 product sectors. First 5:
  Final consumption expenditure by households
  Final consumption expenditure by non-profit organisations serving households (NPISH)
  Final consumption expenditure by government
  Gross fixed capital formation
  Changes in inventories


### 4.2 GHG aggregation (fossil + biogenic)

We aggregate the 420-row `air_emissions` extension into two climate streams:

- **Fossil GHG (kt CO2-eq)** — combustion CO2, cement & lime process CO2, fossil waste CO2, fossil CH4, combustion / agriculture N2O, plus HFC / PFC / SF6.
- **Biogenic GHG (kt CO2-eq)** — `CO2_bio`, `CH4_bio`, `N2O_bio`, peat-decay CO2 and biogenic-waste CO2.

AR5 GWP-100 factors are applied. Unit handling follows EXIOBASE v3.8.2's `air_emissions/unit.txt`:

- CO2, CH4, N2O, SF6 rows are physical kg → multiplied by AR5 GWP factor.
- HFC, PFC rows are already **kg CO2-eq** → GWP factor = 1.

In [6]:
GWP_AR5 = {"CO2": 1, "CH4": 28, "N2O": 265, "SF6": 23500, "HFC": 1, "PFC": 1}

GHG_FOSSIL_ROWS = {
    "CO2": [
        "CO2 - combustion - air",
        "CO2 - non combustion - Cement production - air",
        "CO2 - non combustion - Lime production - air",
        "CO2 - waste - fossil - air",
    ],
    "CH4": [
        "CH4 - combustion - air",
        "CH4 - non combustion - Extraction/production of (natural) gas - air",
        "CH4 - non combustion - Extraction/production of crude oil - air",
        "CH4 - non combustion - Mining of antracite - air",
        "CH4 - non combustion - Mining of bituminous coal - air",
        "CH4 - non combustion - Mining of coking coal - air",
        "CH4 - non combustion - Mining of lignite (brown coal) - air",
        "CH4 - non combustion - Mining of sub-bituminous coal - air",
        "CH4 - non combustion - Oil refinery - air",
        "CH4 - agriculture - air",
        "CH4 - waste - air",
    ],
    "N2O": ["N2O - combustion - air", "N2O - agriculture - air"],
    "SF6": ["SF6 - air"],
    "HFC": ["HFC - air"],
    "PFC": ["PFC - air"],
}

GHG_BIOGENIC_ROWS = {
    "CO2": [
        "CO2_bio - combustion - air",
        "CO2 - agriculture - peat decay - air",
        "CO2 - waste - biogenic - air",
    ],
    "CH4": ["CH4_bio - combustion - air"],
    "N2O": ["N2O_bio - combustion - air"],
}

def _aggregate(flow_matrix, row_dict, label):
    """Aggregate selected air-emission rows into CO2-eq, dividing kg -> kt at the end."""
    present = set(flow_matrix.index.tolist())
    out = None
    missing = []
    for gas, rows in row_dict.items():
        gwp = GWP_AR5[gas]
        for row in rows:
            if row in present:
                contrib = flow_matrix.loc[row] * gwp
                out = contrib if out is None else out + contrib
            else:
                missing.append(row)
    if missing:
        print(f"  [aggregate_ghg::{label}] {len(missing)} rows missing: {missing[:3]}...")
    return out / 1e6   # kg -> kt

def aggregate_ghg_fossil(flow_matrix):
    return _aggregate(flow_matrix, GHG_FOSSIL_ROWS, "fossil")

def aggregate_ghg_biogenic(flow_matrix):
    return _aggregate(flow_matrix, GHG_BIOGENIC_ROWS, "biogenic")


### 4.3 Material aggregation — Anthesis 4 categories

The material extension has 62 rows. We group them positionally:

| Row range | Anthesis category | EXIOBASE content |
|---|---|---|
| 0–22  | biomass  | primary crops, grazing, forestry, fish |
| 23–32 | fossil   | coal, oil, gas, peat |
| 33–47 | metals   | metal ores |
| 48–61 | minerals | non-metallic minerals (stone, sand, chemicals) |

We print the actual row names so the mapping can be audited.

In [21]:
list(exio.get_extensions())

['satellite', 'impacts']

In [22]:
exio.satellite.D_cba.SE

sector,Paddy rice,Wheat,Cereal grains nec,"Vegetables, fruit, nuts",Oil seeds,"Sugar cane, sugar beet",Plant-based fibers,Crops nec,Cattle,Pigs,...,Paper for treatment: landfill,Plastic waste for treatment: landfill,Inert/metal/hazardous waste for treatment: landfill,Textiles waste for treatment: landfill,Wood waste for treatment: landfill,Membership organisation services n.e.c. (91),"Recreational, cultural and sporting services (92)",Other services (93),Private households with employed persons (95),Extra-territorial organizations and bodies
stressor,,,,,,,,,,,,,,,,,,,,,
Taxes less subsidies on products purchased: Total,0.003455,53.935326,8.203488,87.812202,13.024824,5.287795e-06,3.269719e-06,47.755137,6.454258,27.733615,...,0,0,0,0,0,893.653493,645.090401,81.129550,6.085142,0
Other net taxes on production,0.002893,-48.664807,-11.108398,-0.975537,-21.768227,-3.861559e-06,7.950860e-08,-2.196928,-4.640691,-5.477182,...,0,0,0,0,0,569.381344,136.027218,99.181019,4.350784,0
"Compensation of employees; wages, salaries, & employers' social contributions: Low-skilled",0.011460,12.715020,1.349872,72.579484,9.426752,1.064276e-06,1.723054e-06,27.855799,0.961251,3.663769,...,0,0,0,0,0,742.087047,268.673854,69.121144,179.391415,0
"Compensation of employees; wages, salaries, & employers' social contributions: Medium-skilled",0.014939,140.396796,20.851154,377.418119,129.996687,9.096663e-06,9.045010e-06,151.194249,13.620752,49.339726,...,0,0,0,0,0,6532.987544,2138.042579,638.721627,89.625792,0
"Compensation of employees; wages, salaries, & employers' social contributions: High-skilled",0.005618,71.646464,9.541199,160.459011,29.952520,7.916137e-06,2.452121e-06,52.241389,5.959385,26.994052,...,0,0,0,0,0,9537.882145,2964.093587,821.992297,25.741767,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Energy Carrier Net TMAR,0.005306,66.186767,6.498096,96.686067,21.171473,5.943898e-06,2.690008e-06,62.533814,3.895135,17.926801,...,0,0,0,0,0,585.100147,711.153049,135.849021,15.516026,0
Energy Carrier Net TOTH,0.002063,102.532083,10.619479,48.852814,15.144946,6.337898e-06,2.998032e-06,24.492953,3.029200,12.729051,...,0,0,0,0,0,159.556635,159.859869,58.415951,5.066574,0
Energy Carrier Net TRAI,0.000384,7.383099,0.836461,8.747759,1.917900,3.548860e-07,7.206073e-07,3.317662,0.344909,1.505619,...,0,0,0,0,0,95.260636,43.950622,10.527645,0.884780,0


In [28]:
exio.satellite.D_cba.head(20)

region                                                     AT             \
sector                                             Paddy rice      Wheat   
stressor                                                                   
Taxes less subsidies on products purchased: Total    0.022027   9.299229   
Other net taxes on production                        0.010741  -4.493761   
Compensation of employees; wages, salaries, & e...   0.032354   4.385752   
Compensation of employees; wages, salaries, & e...   0.176234  28.084832   
Compensation of employees; wages, salaries, & e...   0.020302  20.004077   
Operating surplus: Consumption of fixed capital      0.066255  34.960890   
Operating surplus: Rents on land                     0.000000   0.000000   
Operating surplus: Royalties on resources            0.000000   0.000000   
Operating surplus: Remaining net operating surplus   0.112054  85.034448   
Employment: Low-skilled male                         0.023737   1.191675   
Employment: Low-skilled female                       0.012877   0.677055   
Employment: Medium-skilled male                      0.125201   5.079330   
Employment: Medium-skilled female                    0.077647   1.960508   
Employment: High-skilled male                        0.002419   1.045589   
Employment: High-skilled female                      0.000409   0.507161   
Employment hours: Low-skilled male                   0.043652   2.480937   
Employment hours: Low-skilled female                 0.022664   1.333847   
Employment hours: Medium-skilled male                0.244232  10.322288   
Employment hours: Medium-skilled female              0.137896   3.800451   
Employment hours: High-skilled male                  0.004807   2.114531   

region                                                                \
sector                                             Cereal grains nec   
stressor                                                               
Taxes less subsidies on products purchased: Total          15.998276   
Other net taxes on production                             -10.517273   
Compensation of employees; wages, salaries, & e...          6.943671   
Compensation of employees; wages, salaries, & e...         48.184710   
Compensation of employees; wages, salaries, & e...         29.387327   
Operating surplus: Consumption of fixed capital            89.044142   
Operating surplus: Rents on land                            0.000000   
Operating surplus: Royalties on resources                   0.000000   
Operating surplus: Remaining net operating surplus        198.534984   
Employment: Low-skilled male                                2.480264   
Employment: Low-skilled female                              1.718421   
Employment: Medium-skilled male                             9.627671   
Employment: Medium-skilled female                           4.604724   
Employment: High-skilled male                               1.305771   
Employment: High-skilled female                             0.645771   
Employment hours: Low-skilled male                          4.956498   
Employment hours: Low-skilled female                        3.214126   
Employment hours: Medium-skilled male                      18.614356   
Employment hours: Medium-skilled female                     8.891134   
Employment hours: High-skilled male                         2.614877   

region                                                                      \
sector                                             Vegetables, fruit, nuts   
stressor                                                                     
Taxes less subsidies on products purchased: Total               109.815671   
Other net taxes on production                                   -19.560790   
Compensation of employees; wages, salaries, & e...               81.451503   
Compensation of employees; wages, salaries, & e...              425.806074   
Compensation of employees; wages, salaries, & e.

In [25]:
# ---------- Diagnostic: what extensions and rows are really in exio? ----------
print("Extensions on exio:")
for name in dir(exio):
    obj = getattr(exio, name)
    if hasattr(obj, 'F') and not name.startswith('_'):
        F = obj.F
        if F is not None:
            print(f"  exio.{name}  F shape: {F.shape}")

print("\n--- Sampling row names in each extension ---")
for name in dir(exio):
    obj = getattr(exio, name)
    if hasattr(obj, 'F') and not name.startswith('_'):
        F = obj.F
        if F is None:
            continue
        rows = F.index.tolist()
        print(f"\n### exio.{name}  ({len(rows)} rows) ###")
        # Show first, last, and a few middle samples
        indices_to_show = [0, 1, 2, len(rows)//4, len(rows)//2, 3*len(rows)//4, len(rows)-2, len(rows)-1]
        for i in sorted(set(indices_to_show)):
            print(f"  [{i:4d}] {rows[i]}")
        # Also search for key patterns
        patterns = ['Domestic Extraction', 'CO2 - combustion', 'Compensation of employees',
                    'Operating surplus', 'Taxes less', 'Value Added', 'Employment']
        for p in patterns:
            hits = [r for r in rows if p.lower() in r.lower()]
            if hits:
                print(f"    -> pattern '{p}' found {len(hits)} row(s), first: {hits[0]!r}")

Extensions on exio:
  exio.impacts  F shape: (126, 9800)
  exio.satellite  F shape: (1113, 9800)

--- Sampling row names in each extension ---

### exio.impacts  (126 rows) ###
  [   0] Value Added
  [   1] Employment
  [   2] Employment hour
  [  31] Water Consumption Blue - Domestic
  [  63] human toxicity (HTP100) | Problem oriented approach: non baseline (CML, 1999) | HTP 100 (Huijbregts, 1999 & 2000)
  [  94] Photochemical ozone formation midpoint, human health | ILCD recommended CF | Photochemical ozone creation potential (POCP)
  [ 124] Unused Domestic Extraction - Non-ferous metal ores
  [ 125] Land use Crop, Forest, Pasture
    -> pattern 'Domestic Extraction' found 17 row(s), first: 'Unused Domestic Extraction'
    -> pattern 'Value Added' found 1 row(s), first: 'Value Added'
    -> pattern 'Employment' found 2 row(s), first: 'Employment'

### exio.satellite  (1113 rows) ###
  [   0] Taxes less subsidies on products purchased: Total
  [   1] Other net taxes on production
  [ 

In [29]:

#Diagnostic 1 — material extraction rows and their units
# Find all "Domestic Extraction Used" (DEU) rows and their units
sat = exio.satellite
deu_rows = [r for r in sat.F.index.tolist() if 'Domestic Extraction Used' in r]
print(f"Total DEU rows: {len(deu_rows)}\n")

# Group by the part after "Domestic Extraction Used - "
from collections import defaultdict
groups = defaultdict(list)
for r in deu_rows:
    # Split into 'Domestic Extraction Used - <GROUP> - <detail>'
    parts = r.split(' - ')
    group = parts[1] if len(parts) > 1 else 'OTHER'
    groups[group].append(r)

for g, rows in groups.items():
    print(f"\n### {g}  ({len(rows)} rows) ###")
    for r in rows[:4]:
        print(f"  {r}")
    if len(rows) > 4:
        print(f"  ... and {len(rows) - 4} more")

# Units for DEU rows (first 5 and any unique units)
print("\n--- Units ---")
if sat.unit is not None:
    units_deu = sat.unit.loc[deu_rows]
    print(f"Unique units across all DEU rows: {units_deu.iloc[:, 0].unique().tolist()}")
else:
    print("  no unit table on satellite extension")

Total DEU rows: 217


### Crop residues  (2 rows) ###
  Domestic Extraction Used - Crop residues - Feed
  Domestic Extraction Used - Crop residues - Straw

### Fishery  (4 rows) ###
  Domestic Extraction Used - Fishery - Aquatic plants
  Domestic Extraction Used - Fishery - Inland waters fish catch
  Domestic Extraction Used - Fishery - Marine fish catch
  Domestic Extraction Used - Fishery - Other (e.g. Aquatic mammals)

### Fodder crops  (16 rows) ###
  Domestic Extraction Used - Fodder crops - Alfalfa for Forage and Silage
  Domestic Extraction Used - Fodder crops - Beets for Fodder
  Domestic Extraction Used - Fodder crops - Cabbage for Fodder
  Domestic Extraction Used - Fodder crops - Carrots for Fodder
  ... and 12 more

### Forestry  (7 rows) ###
  Domestic Extraction Used - Forestry - Coniferous wood - Industrial roundwood
  Domestic Extraction Used - Forestry - Coniferous wood - Wood fuel
  Domestic Extraction Used - Forestry - Kapok Fruit
  Domestic Extraction Used - Forestr

In [ ]:

# Diagnostic 2 — GHG rows, value-added rows, and the impacts extension
# GHG rows in satellite
sat = exio.satellite
ghg_patterns = ['CO2', 'CH4', 'N2O', 'SF6', 'HFC', 'PFC']
ghg_rows = [r for r in sat.F.index.tolist() if any(r.startswith(p) for p in ghg_patterns)]
print(f"GHG-candidate rows in satellite: {len(ghg_rows)}\n")
for r in ghg_rows[:20]:
    print(f"  {r}")
print(f"  ... ({len(ghg_rows)} total)")

# Value-added component rows in satellite
print("\n--- Value-added component rows in satellite ---")
va_patterns = ['Taxes less', 'Other net taxes', 'Compensation of employees', 'Operating surplus']
for p in va_patterns:
    hits = [r for r in sat.F.index.tolist() if r.startswith(p)]
    for r in hits:
        print(f"  {r}")

# Full list of impacts extension rows (all 126 — we may use pre-computed GWP100 if present)
print("\n--- Full impacts extension index (126 rows) ---")
for i, r in enumerate(exio.impacts.F.index.tolist()):
    print(f"  [{i:3d}] {r}")

# Units on impacts
print("\n--- Impacts units (first 10) ---")
if exio.impacts.unit is not None:
    print(exio.impacts.unit.head(10))

GHG-candidate rows in satellite: 22

  CO2 - combustion - air
  CH4 - combustion - air
  N2O - combustion - air
  CH4 - non combustion - Extraction/production of (natural) gas - air
  CH4 - non combustion - Extraction/production of crude oil - air
  CH4 - non combustion - Mining of antracite - air
  CH4 - non combustion - Mining of bituminous coal - air
  CH4 - non combustion - Mining of coking coal - air
  CH4 - non combustion - Mining of lignite (brown coal) - air
  CH4 - non combustion - Mining of sub-bituminous coal - air
  CH4 - non combustion - Oil refinery - air
  CO2 - non combustion - Cement production - air
  CO2 - non combustion - Lime production - air
  SF6 - air
  HFC - air
  PFC - air
  CH4 - agriculture - air
  CO2 - agriculture - peat decay - air
  N2O - agriculture - air
  CH4 - waste - air
  ... (22 total)

--- Value-added component rows in satellite ---
  Taxes less subsidies on products purchased: Total
  Other net taxes on production
  Compensation of employees; wa

In [7]:
MATERIAL_CATEGORY_ROWS = {
    "biomass":  (0, 23),
    "fossil":   (23, 33),
    "metals":   (33, 48),
    "minerals": (48, 62),
}

def build_material_mapping(material_F, verbose=True):
    rows = material_F.index.tolist()
    assert len(rows) == 62, f"Expected 62 material rows, got {len(rows)}"
    mapping = {cat: rows[start:end] for cat, (start, end) in MATERIAL_CATEGORY_ROWS.items()}
    if verbose:
        for cat, rws in mapping.items():
            print(f"  {cat:<9} ({len(rws)} rows):  {rws[0]}  ...  {rws[-1]}")
    return mapping

MATERIAL_MAP = build_material_mapping(exio.material.F)

def aggregate_materials(flow_matrix, mapping=MATERIAL_MAP):
    """Aggregate a material-indexed flow matrix into (4 categories x cols)."""
    out = pd.DataFrame(0.0, index=list(mapping.keys()), columns=flow_matrix.columns)
    for cat, rows in mapping.items():
        rows_present = [r for r in rows if r in flow_matrix.index]
        if rows_present:
            out.loc[cat] = flow_matrix.loc[rows_present].sum(axis=0)
    return out


AttributeError: 'IOSystem' object has no attribute 'material'

### 4.4 Economic aggregation — value added

Value added per sector = sum of all `factor_inputs` rows (labour compensation + taxes − subsidies + operating surplus), in M EUR.

In [ ]:
def aggregate_value_added(factor_matrix):
    """Sum factor_inputs rows to give total value added per sector (M EUR)."""
    return factor_matrix.sum(axis=0)

print("factor_inputs rows included in value-added aggregation:")
for r in exio.factor_inputs.F.index.tolist():
    print(f"  {r}")


## 5. Build sector-level indicator vectors

We compute two sets of vectors for every indicator:

- **F-level** (direct) — impact generated by each producing sector per sector's actual output. Used for PBA.
- **M-level** (multiplier) — total impact embodied per M EUR of final demand delivered by each (origin, product) pair. Used for CBA via `D_cba = M × diag(y_col_sums)`.

In [ ]:
# ---------- Direct impacts (F) ----------
F_ghg_fos  = aggregate_ghg_fossil(exio.air_emissions.F)      # Series, kt CO2e
F_ghg_bio  = aggregate_ghg_biogenic(exio.air_emissions.F)    # Series, kt CO2e
F_mat      = aggregate_materials(exio.material.F)            # DataFrame (4, 9800), kt
F_va       = aggregate_value_added(exio.factor_inputs.F)     # Series, M EUR

print("--- Global direct totals ---")
print(f"  Fossil GHG    : {F_ghg_fos.sum()/1000:>10,.0f} Mt CO2e")
print(f"  Biogenic GHG  : {F_ghg_bio.sum()/1000:>10,.0f} Mt CO2e")
print(f"  Material      : {F_mat.sum().sum()/1000:>10,.0f} Mt")
print(f"  Value Added   : {F_va.sum()/1e6:>10,.1f} T EUR")


In [ ]:
# ---------- Multipliers (M = S @ L) ----------
M_ghg_fos = aggregate_ghg_fossil(exio.air_emissions.M)       # Series, kt CO2e / M EUR
M_ghg_bio = aggregate_ghg_biogenic(exio.air_emissions.M)     # Series, kt CO2e / M EUR
M_mat     = aggregate_materials(exio.material.M)             # DataFrame (4, 9800), kt / M EUR
M_va      = exio.factor_inputs.M.sum(axis=0)                 # Series, M EUR VA / M EUR

# ---------- Direct intensities (S = F / x) ----------
S_ghg_fos = aggregate_ghg_fossil(exio.air_emissions.S)       # Series, kt CO2e / M EUR
S_ghg_bio = aggregate_ghg_biogenic(exio.air_emissions.S)     # Series, kt CO2e / M EUR
S_mat     = aggregate_materials(exio.material.S)             # DataFrame (4, 9800), kt / M EUR
S_va      = exio.factor_inputs.S.sum(axis=0)                 # Series, dimensionless (VA/output)

print(f"M_ghg_fos : {M_ghg_fos.shape}")
print(f"M_mat     : {M_mat.shape}")
print(f"M_va      : {M_va.shape}")


## 6. PBA hotspot analysis — Sweden

Production-based account = direct impacts from Swedish producers, by sector.

In [ ]:
# Direct Swedish impacts per sector (drop the region level so sector becomes the index)
F_ghg_fos_SE = F_ghg_fos.xs(REGION, level="region")
F_ghg_bio_SE = F_ghg_bio.xs(REGION, level="region")
F_mat_SE     = F_mat.xs(REGION, level="region", axis=1)
F_va_SE      = F_va.xs(REGION, level="region")
F_mat_total_SE = F_mat_SE.sum(axis=0)

pba_SE = pd.DataFrame({
    "Value added (M EUR)":     F_va_SE,
    "GHG fossil (kt CO2e)":    F_ghg_fos_SE,
    "GHG biogenic (kt CO2e)":  F_ghg_bio_SE,
    "Biomass (kt)":            F_mat_SE.loc["biomass"],
    "Fossil (kt)":             F_mat_SE.loc["fossil"],
    "Metals (kt)":             F_mat_SE.loc["metals"],
    "Minerals (kt)":           F_mat_SE.loc["minerals"],
    "Material total (kt)":     F_mat_total_SE,
})
pba_SE.index.name = "Product sector"

print("=== Sweden PBA totals ===")
print(f"  Value added        : {F_va_SE.sum()/1000:>10,.1f} B EUR")
print(f"  GHG fossil         : {F_ghg_fos_SE.sum():>10,.0f} kt CO2e")
print(f"  GHG biogenic       : {F_ghg_bio_SE.sum():>10,.0f} kt CO2e")
print(f"  Material total     : {F_mat_total_SE.sum():>10,.0f} kt")

pba_SE.to_csv(OUTPUT_DIR / "pba_sweden_by_sector.csv")


In [ ]:
def show_top(df, col, n=TOP_N):
    return df.nlargest(n, col)

print(f"=== Top {TOP_N} Swedish PBA sectors by VALUE ADDED ===")
display(show_top(pba_SE, "Value added (M EUR)"))


In [ ]:
print(f"=== Top {TOP_N} Swedish PBA sectors by FOSSIL GHG ===")
display(show_top(pba_SE, "GHG fossil (kt CO2e)"))


In [ ]:
print(f"=== Top {TOP_N} Swedish PBA sectors by BIOGENIC GHG ===")
display(show_top(pba_SE, "GHG biogenic (kt CO2e)"))


In [ ]:
print(f"=== Top {TOP_N} Swedish PBA sectors by MATERIAL TOTAL ===")
display(show_top(pba_SE, "Material total (kt)"))


## 7. CBA hotspot analysis — Sweden

Consumption-based account = impacts embodied in Sweden's final demand.

We compute:

$$
D^{CBA}_{SE}[k, (c_{orig}, s)] = M[k, (c_{orig}, s)] \times y^{SE}[(c_{orig}, s)]
$$

where `y^SE` is Sweden's final demand aggregated over final-demand categories (households, government, etc.), and `(c_orig, s)` is the producing (origin country, product sector).

To get Sweden's CBA **by product category consumed** (the conventional hotspot table), we then aggregate across origins:

$$
D^{CBA,\text{product}}_{SE}[k, s] = \sum_{c_{orig}} D^{CBA}_{SE}[k, (c_{orig}, s)]
$$

In [ ]:
# Sweden's final demand, summed across fd categories
Y_SE = cols_of(exio.Y, REGION)          # (9800, 7)  M EUR by (origin, product) x (SE, fd_category)
y_SE = Y_SE.sum(axis=1)                 # (9800,)    M EUR by (origin, product)

# CBA by (origin country, product sector) consumed by Sweden
D_cba_ghg_fos = M_ghg_fos * y_SE                         # (9800,)
D_cba_ghg_bio = M_ghg_bio * y_SE                         # (9800,)
D_cba_mat     = M_mat.multiply(y_SE, axis=1)             # (4, 9800)
D_cba_va      = M_va * y_SE                              # (9800,)

# Aggregate across origins -> CBA per product category Sweden consumed
cba_ghg_fos_by_sec = D_cba_ghg_fos.groupby(level="sector").sum()
cba_ghg_bio_by_sec = D_cba_ghg_bio.groupby(level="sector").sum()
cba_mat_by_sec     = D_cba_mat.T.groupby(level="sector").sum().T    # (4, 200)
cba_va_by_sec      = D_cba_va.groupby(level="sector").sum()
cba_mat_total_by_sec = cba_mat_by_sec.sum(axis=0)

cba_SE = pd.DataFrame({
    "Value added embodied (M EUR)":  cba_va_by_sec,
    "GHG fossil (kt CO2e)":          cba_ghg_fos_by_sec,
    "GHG biogenic (kt CO2e)":        cba_ghg_bio_by_sec,
    "Biomass (kt)":                  cba_mat_by_sec.loc["biomass"],
    "Fossil (kt)":                   cba_mat_by_sec.loc["fossil"],
    "Metals (kt)":                   cba_mat_by_sec.loc["metals"],
    "Minerals (kt)":                 cba_mat_by_sec.loc["minerals"],
    "Material total (kt)":           cba_mat_total_by_sec,
})
cba_SE.index.name = "Product sector consumed"

print("=== Sweden CBA totals ===")
print(f"  Value added embodied : {D_cba_va.sum()/1000:>10,.1f} B EUR")
print(f"  GHG fossil           : {D_cba_ghg_fos.sum():>10,.0f} kt CO2e")
print(f"  GHG biogenic         : {D_cba_ghg_bio.sum():>10,.0f} kt CO2e")
print(f"  Material total       : {D_cba_mat.sum().sum():>10,.0f} kt")

cba_SE.to_csv(OUTPUT_DIR / "cba_sweden_by_sector.csv")


In [ ]:
print(f"=== Top {TOP_N} CBA product sectors in Swedish consumption — VALUE ADDED embodied ===")
display(show_top(cba_SE, "Value added embodied (M EUR)"))


In [ ]:
print(f"=== Top {TOP_N} CBA product sectors in Swedish consumption — FOSSIL GHG ===")
display(show_top(cba_SE, "GHG fossil (kt CO2e)"))


In [ ]:
print(f"=== Top {TOP_N} CBA product sectors in Swedish consumption — MATERIAL TOTAL ===")
display(show_top(cba_SE, "Material total (kt)"))


## 8. Source country / sector attribution for Sweden's CBA

For each top consumption product we want to know **where** the impacts are physically generated. The formula is:

$$
\text{source}[k, (c_{src}, s_{src})] = S[k, (c_{src}, s_{src})] \cdot x^{SE}[(c_{src}, s_{src})]
$$

where

$$
x^{SE} = L \cdot y^{SE}
$$

is the total global output (per producing (country, sector)) driven by Sweden's final demand.

This **aggregates** across consumption sectors. For per-consumption-sector decomposition we build a matrix of "Swedish demand for each product type" and solve once, giving a (9800 producers × 200 consumption sectors) attribution matrix.

In [ ]:
# ---------- Total source attribution (aggregate across all of Sweden's consumption) ----------
t0 = time.time()
x_SE_total = exio.L @ y_SE           # Series, (9800,)  M EUR of global output driven by SE demand
print(f"x_SE_total computed in {time.time()-t0:.1f}s. Shape: {x_SE_total.shape}")

# Source contributions per (origin country, origin sector)
src_ghg_fos = S_ghg_fos * x_SE_total                       # (9800,)  kt CO2e
src_ghg_bio = S_ghg_bio * x_SE_total                       # (9800,)  kt CO2e
src_mat     = S_mat.multiply(x_SE_total, axis=1)           # (4, 9800) kt
src_va      = S_va * x_SE_total                            # (9800,)  M EUR
src_mat_total = src_mat.sum(axis=0)                        # (9800,)  kt

# Sanity check: source totals should equal CBA totals
print(f"\n--- Sanity check (totals must match) ---")
print(f"  GHG fossil : CBA={D_cba_ghg_fos.sum():,.0f}, source sum={src_ghg_fos.sum():,.0f}")
print(f"  GHG bio    : CBA={D_cba_ghg_bio.sum():,.0f}, source sum={src_ghg_bio.sum():,.0f}")
print(f"  Material   : CBA={D_cba_mat.sum().sum():,.0f}, source sum={src_mat.sum().sum():,.0f}")
print(f"  Value added: CBA={D_cba_va.sum():,.0f}, source sum={src_va.sum():,.0f}")


In [ ]:
# ---------- Aggregate by source country ----------
def src_country(s):
    return s.groupby(level="region").sum().sort_values(ascending=False)

src_ghg_fos_country = src_country(src_ghg_fos)
src_ghg_bio_country = src_country(src_ghg_bio)
src_va_country      = src_country(src_va)
src_mat_country     = src_country(src_mat_total)

# Combined country table
src_country_all = pd.DataFrame({
    "Value added (M EUR)":     src_va_country,
    "GHG fossil (kt CO2e)":    src_ghg_fos_country,
    "GHG biogenic (kt CO2e)":  src_ghg_bio_country,
    "Material total (kt)":     src_mat_country,
})
src_country_all.index.name = "Source country"
src_country_all.to_csv(OUTPUT_DIR / "cba_source_by_country.csv")

print(f"=== Top {TOP_N} source countries for Sweden's CBA ===")
display(src_country_all.head(TOP_N))

# Print % share
print("\n--- Sweden's own share (domestic content of CBA) ---")
for col in src_country_all.columns:
    tot = src_country_all[col].sum()
    se  = src_country_all.loc[REGION, col] if REGION in src_country_all.index else 0
    print(f"  {col:<30}: {se/tot*100:5.1f}%")


In [ ]:
# ---------- Top (country, sector) source pairs ----------
def pairs_table(series, n=TOP_N):
    return series.nlargest(n).to_frame("value")

print(f"=== Top {TOP_N} (origin country, origin sector) pairs — GHG fossil ===")
display(pairs_table(src_ghg_fos))


In [ ]:
print(f"=== Top {TOP_N} (origin country, origin sector) pairs — Material total ===")
display(pairs_table(src_mat_total))


In [ ]:
print(f"=== Top {TOP_N} (origin country, origin sector) pairs — Value added ===")
display(pairs_table(src_va))


### 8.1 Per-consumption-sector source decomposition

For each top Swedish consumption product, we attribute impacts back to their origin (country, sector). This is what the workshop audience typically wants: "For Sweden's demand for **food**, where do the impacts come from?"

Approach: build a Swedish-demand matrix of shape (9800 producers × 200 consumed-product-categories), where column `s` carries Sweden's demand for product `s` (from any origin) and zeros elsewhere. One L-multiplication gives the supply-chain output for each consumed product category. Multiplying by S gives the (producer × consumed-product) attribution matrix.

In [ ]:
sectors_list = exio.get_sectors().tolist()
n_sec = len(sectors_list)
print(f"Building per-consumption-sector Y matrix ({n_sec} columns)...")

# Y_SE_by_consumed[producer_idx, consumed_sector] = y_SE[(origin, product)] if product == consumed_sector else 0
Y_SE_by_consumed = pd.DataFrame(0.0, index=exio.Y.index, columns=sectors_list)
for s in sectors_list:
    mask_idx = pd.IndexSlice[:, s]
    Y_SE_by_consumed.loc[mask_idx, s] = y_SE.loc[mask_idx].values

t0 = time.time()
# Single big L @ Y matrix multiplication
X_SE_by_consumed = pd.DataFrame(
    exio.L.values @ Y_SE_by_consumed.values,
    index=exio.L.index, columns=sectors_list
)
print(f"  X_SE_by_consumed computed in {time.time()-t0:.1f}s. Shape: {X_SE_by_consumed.shape}")

# Attribution matrices: rows = (origin country, origin sector), cols = consumed product sector
attrib_ghg_fos = X_SE_by_consumed.multiply(S_ghg_fos, axis=0)                   # (9800, 200)
attrib_ghg_bio = X_SE_by_consumed.multiply(S_ghg_bio, axis=0)                   # (9800, 200)
attrib_va      = X_SE_by_consumed.multiply(S_va, axis=0)                        # (9800, 200)
attrib_mat_total = X_SE_by_consumed.multiply(S_mat.sum(axis=0), axis=0)           # (9800, 200)

print(f"attrib_ghg_fos  : {attrib_ghg_fos.shape}")
print(f"attrib_mat_total: {attrib_mat_total.shape}")


## 9. Destination attribution for Sweden's PBA

Where does Sweden's production end up? Each Swedish sector's output is used by some chain of other sectors and ultimately satisfies final demand in some country. We decompose Sweden's direct emissions:

$$
F^{SE}[k, i, (c_{cons}, cat)] = S^{SE}[k, i] \times \big(L^{SE \text{-rows}} \cdot Y\big)[i, (c_{cons}, cat)]
$$

The key operation is `L[SE rows, :] @ Y`, which gives Swedish gross output by Swedish sector × consumer (country, fd category). Multiplying by Swedish direct intensity and aggregating gives the destination breakdown.

In [ ]:
t0 = time.time()
L_SE_rows = exio.L.xs(REGION, level="region")                              # (200, 9800)
x_SE_dest = pd.DataFrame(
    L_SE_rows.values @ exio.Y.values,
    index=L_SE_rows.index,                                                  # SE sector
    columns=exio.Y.columns                                                  # (consumer_region, fd_cat)
)
print(f"x_SE_dest computed in {time.time()-t0:.1f}s. Shape: {x_SE_dest.shape}")

# Swedish direct intensity vectors (one per SE sector)
S_ghg_fos_SE = S_ghg_fos.xs(REGION, level="region")
S_ghg_bio_SE = S_ghg_bio.xs(REGION, level="region")
S_mat_SE     = S_mat.xs(REGION, level="region", axis=1)
S_va_SE      = S_va.xs(REGION, level="region")

# F_SE per destination column
F_ghg_fos_dest = x_SE_dest.multiply(S_ghg_fos_SE, axis=0)                 # (200, n_dest)
F_ghg_bio_dest = x_SE_dest.multiply(S_ghg_bio_SE, axis=0)                 # (200, n_dest)
F_va_dest      = x_SE_dest.multiply(S_va_SE, axis=0)                       # (200, n_dest)
F_mat_dest_total = x_SE_dest.multiply(S_mat_SE.sum(axis=0), axis=0)       # (200, n_dest)

# Sanity check vs direct PBA
print(f"\n--- Sanity check (must match Sweden PBA totals) ---")
print(f"  GHG fossil   : PBA={F_ghg_fos_SE.sum():,.0f}, via destinations={F_ghg_fos_dest.sum().sum():,.0f}")
print(f"  GHG biogenic : PBA={F_ghg_bio_SE.sum():,.0f}, via destinations={F_ghg_bio_dest.sum().sum():,.0f}")
print(f"  Material tot : PBA={F_mat_total_SE.sum():,.0f}, via destinations={F_mat_dest_total.sum().sum():,.0f}")
print(f"  Value added  : PBA={F_va_SE.sum():,.0f}, via destinations={F_va_dest.sum().sum():,.0f}")


In [ ]:
# Aggregate destinations to country level (sum across final-demand categories within each country)
def dest_country(mat):
    return mat.T.groupby(level="region").sum().T.sum(axis=0).sort_values(ascending=False)

ghg_fos_dest_country = dest_country(F_ghg_fos_dest)
ghg_bio_dest_country = dest_country(F_ghg_bio_dest)
va_dest_country      = dest_country(F_va_dest)
mat_dest_country     = dest_country(F_mat_dest_total)

dest_country_all = pd.DataFrame({
    "Value added (M EUR)":     va_dest_country,
    "GHG fossil (kt CO2e)":    ghg_fos_dest_country,
    "GHG biogenic (kt CO2e)":  ghg_bio_dest_country,
    "Material total (kt)":     mat_dest_country,
})
dest_country_all.index.name = "Consumer country"
dest_country_all.to_csv(OUTPUT_DIR / "pba_destination_by_country.csv")

print(f"=== Top {TOP_N} destination countries for Sweden's PBA ===")
display(dest_country_all.head(TOP_N))

print("\n--- Share of Sweden's PBA retained domestically vs exported ---")
for col in dest_country_all.columns:
    tot = dest_country_all[col].sum()
    se  = dest_country_all.loc[REGION, col] if REGION in dest_country_all.index else 0
    print(f"  {col:<30}: {se/tot*100:5.1f}% stays in Sweden, {(tot-se)/tot*100:5.1f}% exported")


In [ ]:
# Aggregate destinations to fd category (sum across countries)
def dest_by_category(mat):
    return mat.T.groupby(level="category").sum().T.sum(axis=0).sort_values(ascending=False)

ghg_fos_by_cat = dest_by_category(F_ghg_fos_dest)
va_by_cat      = dest_by_category(F_va_dest)
mat_by_cat     = dest_by_category(F_mat_dest_total)

print("=== Sweden's PBA by final-demand CATEGORY (global aggregation) ===")
dest_cat_all = pd.DataFrame({
    "Value added (M EUR)":  va_by_cat,
    "GHG fossil (kt CO2e)": ghg_fos_by_cat,
    "Material total (kt)":  mat_by_cat,
})
dest_cat_all.index.name = "Final-demand category"
display(dest_cat_all)


## 10. Net trade picture — PBA vs CBA

In [ ]:
net = pd.DataFrame({
    "PBA":  [F_va_SE.sum(), F_ghg_fos_SE.sum(), F_ghg_bio_SE.sum(), F_mat_total_SE.sum()],
    "CBA":  [D_cba_va.sum(), D_cba_ghg_fos.sum(), D_cba_ghg_bio.sum(), D_cba_mat.sum().sum()],
}, index=["Value added (M EUR)", "GHG fossil (kt CO2e)", "GHG biogenic (kt CO2e)", "Material total (kt)"])
net["Net trade (PBA - CBA)"] = net["PBA"] - net["CBA"]
net["CBA / PBA ratio"]        = net["CBA"] / net["PBA"]

print("=== Sweden — PBA vs CBA ===")
display(net)

print("\nInterpretation:")
print("  CBA > PBA  (ratio > 1) -> Sweden is a NET IMPORTER of this impact")
print("  CBA < PBA  (ratio < 1) -> Sweden is a NET EXPORTER of this impact")

net.to_csv(OUTPUT_DIR / "sweden_pba_vs_cba.csv")


## 11. Visualisations

### 11.1 Top sectors — PBA vs CBA side-by-side

In [ ]:
# Compact sector labels (EXIOBASE names are long)
def trim(label, n=45):
    return label if len(label) <= n else label[:n-1] + "…"

def plot_top_horiz(df, col, ax, title, xlabel, color, n=10):
    top = df.nlargest(n, col).iloc[::-1]
    labels = [trim(s) for s in top.index]
    ax.barh(labels, top[col], color=color, edgecolor="white")
    ax.set_title(title, fontsize=10, weight="bold")
    ax.set_xlabel(xlabel, fontsize=9)
    ax.grid(axis="x", alpha=0.3)
    ax.tick_params(labelsize=8)

dims = [
    ("Value added (M EUR)",    "Value added (M EUR)",   "#3a6c8c"),
    ("GHG fossil (kt CO2e)",   "kt CO2e (fossil)",      "#c66d3c"),
    ("GHG biogenic (kt CO2e)", "kt CO2e (biogenic)",    "#a6823a"),
    ("Material total (kt)",    "kt (material total)",   "#6b9b37"),
]

fig, axes = plt.subplots(4, 2, figsize=(15, 18))
for i, (col, xlab, color) in enumerate(dims):
    plot_top_horiz(pba_SE, col, axes[i, 0], f"PBA — top 10 by {col}", xlab, color)
    plot_top_horiz(cba_SE, col, axes[i, 1], f"CBA — top 10 by {col}", xlab, color)
fig.suptitle(f"Sweden hotspot analysis — PBA (left) vs CBA (right)",
             fontsize=13, weight="bold", y=0.995)
fig.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_top_sectors_pba_cba.png", bbox_inches="tight")
plt.show()


### 11.2 Material breakdown per sector (stacked bars)

In [ ]:
def plot_material_stack(top_sectors, mat_df, title, filename):
    df = mat_df.T.loc[top_sectors].iloc[::-1]
    fig, ax = plt.subplots(figsize=(11, 6))
    colors = {"biomass": "#6b9b37", "fossil": "#c66d3c", "metals": "#7a6f91", "minerals": "#c8a850"}
    left = np.zeros(len(df))
    labels = [trim(s) for s in df.index]
    for cat in ["biomass", "fossil", "metals", "minerals"]:
        ax.barh(labels, df[cat], left=left, color=colors[cat], label=cat.capitalize(),
                edgecolor="white", linewidth=0.5)
        left += df[cat].values
    ax.set_xlabel("kt")
    ax.set_title(title, weight="bold")
    ax.legend(loc="lower right")
    ax.grid(axis="x", alpha=0.3)
    fig.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, bbox_inches="tight")
    plt.show()

top_pba_mat = pba_SE.nlargest(12, "Material total (kt)").index.tolist()
plot_material_stack(top_pba_mat, F_mat_SE, "Sweden PBA — top sectors, material composition",
                    "fig_material_stack_pba.png")

top_cba_mat = cba_SE.nlargest(12, "Material total (kt)").index.tolist()
plot_material_stack(top_cba_mat, cba_mat_by_sec, "Sweden CBA — top sectors, material composition",
                    "fig_material_stack_cba.png")


### 11.3 Domestic vs imported / exported shares

In [ ]:
# CBA side (origin composition)
def domestic_imported(series_country):
    dom = series_country.loc[REGION] if REGION in series_country.index else 0
    tot = series_country.sum()
    return dom, tot - dom

dims_plot = [
    ("Value added",  src_va_country,      va_dest_country),
    ("GHG fossil",   src_ghg_fos_country, ghg_fos_dest_country),
    ("GHG biogenic", src_ghg_bio_country, ghg_bio_dest_country),
    ("Material",     src_mat_country,     mat_dest_country),
]

cba_vals, pba_vals, names = [], [], []
for name, cba_ser, pba_ser in dims_plot:
    names.append(name)
    d, i_ = domestic_imported(cba_ser); cba_vals.append((d, i_))
    d, e  = domestic_imported(pba_ser); pba_vals.append((d, e))

cba_arr = np.array(cba_vals, dtype=float)
pba_arr = np.array(pba_vals, dtype=float)
cba_pct = cba_arr / cba_arr.sum(axis=1, keepdims=True) * 100
pba_pct = pba_arr / pba_arr.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for ax, pct, side_labels, title in [
    (axes[0], cba_pct, ("Domestic (SE)", "Imported"),  "CBA — origin composition"),
    (axes[1], pba_pct, ("Stays in SE",   "Exported"),  "PBA — destination composition"),
]:
    ax.barh(names, pct[:, 0], color="#3a6c8c", label=side_labels[0])
    ax.barh(names, pct[:, 1], left=pct[:, 0], color="#c66d3c", label=side_labels[1])
    ax.set_xlim(0, 100)
    ax.set_xlabel("% share")
    ax.set_title(title, weight="bold")
    ax.legend(loc="lower right")
    for i, (d, o) in enumerate(pct):
        if d > 6:
            ax.text(d/2, i, f"{d:.0f}%", ha="center", va="center", color="white", weight="bold", fontsize=9)
        if o > 6:
            ax.text(d + o/2, i, f"{o:.0f}%", ha="center", va="center", color="white", weight="bold", fontsize=9)

fig.suptitle(f"Sweden — domestic vs international share", fontsize=13, weight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(OUTPUT_DIR / "fig_domestic_vs_international.png", bbox_inches="tight")
plt.show()


### 11.4 PBA vs CBA — paired comparison per sector (top 15 by total CBA impact)

In [ ]:
def paired_pba_cba(col_name, color_pba, color_cba, title, filename, n=15):
    # Use the larger of the two to pick the top sectors
    combined = pd.DataFrame({"PBA": pba_SE[col_name], "CBA": cba_SE[col_name]}).fillna(0)
    combined["sum"] = combined["PBA"] + combined["CBA"]
    top = combined.nlargest(n, "sum").drop(columns="sum").iloc[::-1]

    fig, ax = plt.subplots(figsize=(12, 7))
    y = np.arange(len(top))
    h = 0.38
    ax.barh(y - h/2, top["PBA"], height=h, label="PBA (production)",  color=color_pba, edgecolor="white")
    ax.barh(y + h/2, top["CBA"], height=h, label="CBA (consumption)", color=color_cba, edgecolor="white")
    ax.set_yticks(y)
    ax.set_yticklabels([trim(s) for s in top.index])
    ax.set_xlabel(col_name)
    ax.set_title(title, weight="bold")
    ax.legend(loc="lower right")
    ax.grid(axis="x", alpha=0.3)
    fig.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, bbox_inches="tight")
    plt.show()

paired_pba_cba("GHG fossil (kt CO2e)",   "#b06a3f", "#c66d3c",
               "Sweden — PBA vs CBA, fossil GHG, top 15 sectors",
               "fig_paired_ghg.png")
paired_pba_cba("Material total (kt)",    "#547e2a", "#6b9b37",
               "Sweden — PBA vs CBA, material total, top 15 sectors",
               "fig_paired_material.png")
paired_pba_cba("Value added (M EUR)" if "Value added (M EUR)" in pba_SE.columns else pba_SE.columns[0],
               "#2d5670", "#3a6c8c",
               "Sweden — PBA vs CBA, value added, top 15 sectors",
               "fig_paired_va.png")


### 11.5 Sankey — CBA source flows

Left column: top (origin country, origin sector) pairs driving Sweden's CBA.
Right column: top Swedish consumption products.

In [ ]:
def sankey_source_to_consumption(attrib, title, filename, n_src=15, n_cons=10,
                                  unit_label="", color_left="#3a6c8c", color_right="#c66d3c"):
    """
    attrib : DataFrame (9800 origin rows, 200 consumption columns)
    """
    # Top origin (country, sector) pairs by row sum
    top_src  = attrib.sum(axis=1).nlargest(n_src).index
    top_cons = attrib.sum(axis=0).nlargest(n_cons).index

    sub = attrib.loc[top_src, top_cons]

    left_labels  = [f"{c} — {trim(s, 30)}" for (c, s) in top_src]
    right_labels = [trim(s, 40) for s in top_cons]
    nodes = left_labels + right_labels

    colors = [color_left] * len(left_labels) + [color_right] * len(right_labels)

    sources, targets, values = [], [], []
    for i, src_idx in enumerate(top_src):
        for j, cons in enumerate(top_cons):
            v = sub.loc[src_idx, cons]
            if v > 0:
                sources.append(i)
                targets.append(len(left_labels) + j)
                values.append(float(v))

    fig = go.Figure(data=[go.Sankey(
        arrangement="snap",
        node=dict(label=nodes, pad=14, thickness=16, color=colors,
                  line=dict(color="white", width=0.5)),
        link=dict(source=sources, target=targets, value=values,
                  color="rgba(120,120,120,0.25)")
    )])
    fig.update_layout(
        title=dict(text=f"{title}  ({unit_label})", font=dict(size=14)),
        font=dict(size=10), height=650, margin=dict(l=10, r=10, t=55, b=10))
    fig.write_html(str(OUTPUT_DIR / filename))
    fig.show()
    return fig

sankey_source_to_consumption(attrib_ghg_fos,
    "Sweden CBA (fossil GHG) — origin (country, sector) → Swedish consumption",
    "sankey_ghg_source.html", unit_label="kt CO2e")

sankey_source_to_consumption(attrib_mat_total,
    "Sweden CBA (material total) — origin (country, sector) → Swedish consumption",
    "sankey_material_source.html", unit_label="kt")

sankey_source_to_consumption(attrib_va,
    "Sweden CBA (value added) — origin (country, sector) → Swedish consumption",
    "sankey_va_source.html", unit_label="M EUR")


### 11.6 Sankey — PBA destination flows

Left column: top Swedish producing sectors (by PBA).
Right column: destination countries (including Sweden itself, for the retained share).

In [ ]:
def sankey_sector_to_destination(F_dest, pba_series, title, filename, n_sec=12, n_dest=10,
                                  unit_label="", color_left="#6b9b37", color_right="#3a6c8c"):
    """
    F_dest      : DataFrame (200 SE sectors x n_dest consumer columns),
                   columns MultiIndex (consumer_region, fd_category)
    pba_series  : Series (200,) direct PBA per Swedish sector (for ranking)
    """
    # Aggregate destinations to country
    F_dest_country = F_dest.T.groupby(level="region").sum().T   # (200 SE sectors, 49 countries)

    top_secs  = pba_series.nlargest(n_sec).index
    top_dests = F_dest_country.sum(axis=0).nlargest(n_dest).index
    # Ensure Sweden is included if not already
    if REGION not in top_dests:
        top_dests = top_dests.tolist() + [REGION]

    sub = F_dest_country.loc[top_secs, top_dests]

    left_labels  = [trim(s, 40) for s in top_secs]
    right_labels = list(top_dests)
    nodes = left_labels + right_labels

    # Highlight Sweden destination in a different colour
    right_colors = ["#a84020" if c == REGION else color_right for c in right_labels]
    colors = [color_left] * len(left_labels) + right_colors

    sources, targets, values = [], [], []
    for i, s in enumerate(top_secs):
        for j, c in enumerate(top_dests):
            v = sub.loc[s, c]
            if v > 0:
                sources.append(i)
                targets.append(len(left_labels) + j)
                values.append(float(v))

    fig = go.Figure(data=[go.Sankey(
        arrangement="snap",
        node=dict(label=nodes, pad=14, thickness=16, color=colors,
                  line=dict(color="white", width=0.5)),
        link=dict(source=sources, target=targets, value=values,
                  color="rgba(120,120,120,0.25)")
    )])
    fig.update_layout(
        title=dict(text=f"{title}  ({unit_label})", font=dict(size=14)),
        font=dict(size=10), height=650, margin=dict(l=10, r=10, t=55, b=10))
    fig.write_html(str(OUTPUT_DIR / filename))
    fig.show()
    return fig

sankey_sector_to_destination(F_ghg_fos_dest, F_ghg_fos_SE,
    "Sweden PBA (fossil GHG) — Swedish sector → consumer country",
    "sankey_ghg_destination.html", unit_label="kt CO2e")

sankey_sector_to_destination(F_mat_dest_total, F_mat_total_SE,
    "Sweden PBA (material total) — Swedish sector → consumer country",
    "sankey_material_destination.html", unit_label="kt")

sankey_sector_to_destination(F_va_dest, F_va_SE,
    "Sweden PBA (value added) — Swedish sector → consumer country",
    "sankey_va_destination.html", unit_label="M EUR")


### 11.7 Heatmap — Country × sector source attribution (top CBA contributors)

For GHG and material, this shows the concentration of CBA hotspots in the global (country × source sector) grid.

In [ ]:
def heatmap_source(src_series, title, filename, n_countries=12, n_sectors=15, unit=""):
    """src_series: Series indexed by (region, sector)."""
    # Unstack into country × sector
    m = src_series.unstack(level="sector").fillna(0)     # (49 countries, 200 sectors)
    top_c = m.sum(axis=1).nlargest(n_countries).index
    top_s = m.sum(axis=0).nlargest(n_sectors).index
    sub = m.loc[top_c, top_s]

    fig, ax = plt.subplots(figsize=(13, 6))
    sns.heatmap(sub, cmap="YlOrRd", annot=True, fmt=".0f", linewidths=0.3,
                cbar_kws={"label": unit}, ax=ax, annot_kws={"size": 7})
    ax.set_title(title, weight="bold")
    ax.set_xlabel("Source sector")
    ax.set_ylabel("Source country")
    ax.set_xticklabels([trim(t.get_text(), 30) for t in ax.get_xticklabels()], rotation=35, ha="right")
    fig.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, bbox_inches="tight")
    plt.show()

heatmap_source(src_ghg_fos,
    "Sweden CBA — fossil GHG source attribution (top 12 countries x top 15 source sectors)",
    "fig_heatmap_ghg.png", unit="kt CO2e")

heatmap_source(src_mat_total,
    "Sweden CBA — material source attribution (top 12 countries x top 15 source sectors)",
    "fig_heatmap_material.png", unit="kt")

heatmap_source(src_va,
    "Sweden CBA — value added source attribution (top 12 countries x top 15 source sectors)",
    "fig_heatmap_va.png", unit="M EUR")


### 11.8 Sunburst — CBA composition (hierarchical view)

Nests the view: category → region → top sector. Good companion to the Sankey for workshops.

In [ ]:
def sunburst_cba(src_series, title, filename, n_country=10, n_sector=6, unit=""):
    df = src_series.reset_index()
    df.columns = ["region", "sector", "value"]
    df = df[df["value"] > 0]

    # Keep top N countries, aggregate rest as "Other"
    top_c = df.groupby("region")["value"].sum().nlargest(n_country).index
    df["country_label"] = df["region"].where(df["region"].isin(top_c), "Other countries")

    # Within each country keep top N sectors, rest "Other sectors"
    def trim_sectors(sub):
        top_s = sub.groupby("sector")["value"].sum().nlargest(n_sector).index
        sub = sub.copy()
        sub["sector_label"] = sub["sector"].where(sub["sector"].isin(top_s), "Other sectors")
        return sub
    df = df.groupby("country_label", group_keys=False).apply(trim_sectors)
    df_sun = df.groupby(["country_label", "sector_label"])["value"].sum().reset_index()
    df_sun["sector_label"] = df_sun["sector_label"].apply(lambda s: trim(s, 35))

    fig = px.sunburst(
        df_sun, path=["country_label", "sector_label"], values="value",
        color="value", color_continuous_scale="Viridis",
        title=f"{title}  ({unit})",
    )
    fig.update_layout(height=650, margin=dict(l=10, r=10, t=55, b=10))
    fig.write_html(str(OUTPUT_DIR / filename))
    fig.show()
    return fig

sunburst_cba(src_ghg_fos,
    "Sweden CBA — fossil GHG, country and source sector",
    "sunburst_ghg_source.html", unit="kt CO2e")

sunburst_cba(src_mat_total,
    "Sweden CBA — material total, country and source sector",
    "sunburst_material_source.html", unit="kt")


## 12. Export results

In [ ]:
# Summary of what is in ./outputs/
print(f"Outputs written to {OUTPUT_DIR.resolve()}")
for p in sorted(OUTPUT_DIR.iterdir()):
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:<45}  {size_kb:>8,.1f} KB")


---

## Notes for the Stockholm disaggregation step

When the Stockholm / Rest-of-Sweden split is layered on top of this notebook, the structure stays the same:

1. The column-copy disaggregation changes only the (region, sector) MultiIndex of every matrix (adds an `STO` region alongside `SE`). All downstream slicing (`REGION = "SE"`) generalises naturally to `REGION = "STO"`.
2. The Leontief inverse must be recomputed after disaggregation (since `A` changes).
3. All aggregations (`aggregate_ghg_*`, `aggregate_materials`, `aggregate_value_added`) are index-robust and need no changes.
4. The known methodological caveat stands: proportional column-copy disaggregation preserves row and column sums but distorts the CBA / PBA ratio for regions much smaller than their parent. This should be reported alongside the Stockholm results.
